# TESS: configure a task, preview a run, inspect its results

Predict a star's near-core rotation frequency from its light curve. You will inspect the real task package, adjust a small demo, and launch from a **terminal script**. The notebook does not train a model itself.

**Before opening:** follow `tutorials/paper/README.md` in the exp repository. Use an external notebook copy, the **SIDERIUS exp tutorial** kernel, and the exported `EXP_CHECKOUT`, `INFRA_CHECKOUT`, `TUTORIAL_HOME` paths.

Default **Run All** is read-only and offline. It starts no paid API call or GPU run. The two file-writing exercises below require explicit opt-in. The demo is not the six-hour paper search or a promise of its score.

In [ ]:
import csv
import json
import os
import shlex
import sys
from pathlib import Path

EXP = Path(os.environ["EXP_CHECKOUT"]).resolve()
INFRA = Path(os.environ["INFRA_CHECKOUT"]).resolve()
HOME = Path(os.environ["TUTORIAL_HOME"]).resolve()
assert Path(sys.prefix).resolve() == (EXP / ".venv").resolve(), (
    "Select the exp tutorial kernel"
)
assert not HOME.is_relative_to(EXP) and not EXP.is_relative_to(HOME)
assert not HOME.is_relative_to(INFRA) and not INFRA.is_relative_to(HOME)
# package=false: use the checkout cwd, never another checkout through PYTHONPATH.
os.chdir(EXP)
from tutorials.paper.runner import DemoSettings, build_command

TASK = EXP / "tasks/phyts_tess"
WORKFLOW = EXP / "experiments/phyts_tess/main_fixed_workflow"
DATA = HOME / "tess-data"
print("Exp interpreter:", sys.executable)
print("External tutorial directory:", HOME)

## 1. Follow the files from task to run

The **composition** points to the data adapter, scientific declarations, metric and Health checks. The **workflow** chooses iterations, rounds and budgets. The **information treatment** decides which agent modules receive information. Changing one does not implicitly change the others.

| File | Owns |
|---|---|
| `compositions/rotation_regression.yaml` | Links the task's declarations and plugins |
| `declared/task_config.yaml` | Task description and model input/output contract |
| `data/manifests/rotation_identity.csv` | Exact train/validation row identities and labels |
| `runtime/tess_data_path.py` | Normalization, scope selection, tensor materialization |
| `declared/metric_r2.json` | Primary ranking metric; larger R² is better |
| `declared/task_health.yaml` | Checks on model outputs |
| experiment `workflow.json` | Runtime schedule and budgets |
| experiment `agents.json` | Provider/model routing |

The model sees float32 `[B, 1, 1024]` input and returns `[B, 1]` rotation frequency in cycles/day. `B` is batch size. The input length is a task contract, not a free demo knob.

In [ ]:
import yaml

composition = yaml.safe_load(
    (TASK / "compositions/rotation_regression.yaml").read_text()
)
print(yaml.safe_dump(composition, sort_keys=False))
workflow = json.loads((WORKFLOW / "workflow.json").read_text())
print("Paper workflow parameters:")
print(json.dumps(workflow["parameters"], indent=2))

## 2. Understand which data the model may use

One row is a star observed in one sector. The released split has **3,338 training** and **442 validation** curves; the held-out **403 test** curves are not part of the run. Train/validation feedback does not establish independent test performance.

The next cell reads committed identities, not the external raw dataset. The task fixes preprocessing: truncate long curves, normalize observed values, then pad short curves with their last normalized value. Training targets are available for supervised learning; validation remains the scoring population.

In [ ]:
with (TASK / "data/manifests/rotation_identity.csv").open() as stream:
    rows = list(csv.DictReader(stream))
counts = {
    split: sum(row["split"] == split for row in rows) for split in ("train", "val")
}
assert counts == {"train": 3338, "val": 442}
print(counts)
print("Identity columns:", list(rows[0]))

### Download and stage from a terminal

This explicit command downloads **only** the pinned train/validation Parquet files, checks SHA-256, and calls the existing staging tool. It writes external data, but makes no LLM request. The destinations must be new. The source receipt stays in `raw-tess`; only the two NPZ archives enter `tess-data`.

In [ ]:
prepare_command = [
    str(EXP / ".venv/bin/python"),
    "-m",
    "tutorials.paper.prepare_tess",
    "--raw-dir",
    str(HOME / "raw-tess"),
    "--data-dir",
    str(DATA),
]
print("Run from the exp checkout in a terminal:")
print(shlex.join(prepare_command))

### Inspect one actual model input (after staging)

The task implementation below is the normalization authority. The plot is illustrative; it does not choose a training subset or change the data.

In [ ]:
import numpy as np

from tasks.phyts_tess.runtime.tess_data_path import normalize_curve

if (DATA / "tess_rotation_train.npz").is_file():
    import matplotlib.pyplot as plt

    with np.load(DATA / "tess_rotation_train.npz", allow_pickle=False) as archive:
        key = archive.files[0]
        raw = archive[key]
    prepared = normalize_curve(raw, 1024)
    print("Raw length:", len(raw), "model input:", prepared[None, None, :].shape)
    plt.plot(prepared)
    plt.xlabel("Time-sample index")
    plt.ylabel("Normalized flux")
    plt.show()
else:
    print("Data not staged. Run the printed preparation command, then rerun this cell.")

## 3. Check the information treatment and API requirements

TESS NoPrior disables **advice, Data Analysis and literature review**. That differs from the TIDMAD NoPrior treatment. No Semantic Scholar key is needed here.

The shipped active agents use `OPENAI_API_KEY`. In the **launching terminal**, use:

```bash
read -r -s -p "OpenAI API key: " OPENAI_API_KEY
echo
export OPENAI_API_KEY
```

Or load a trusted mode-600 external environment file. Do not put keys in notebook cells, JSON settings, output, or Git. Notebook and terminal environments can differ. Presence is not a provider-access test; this notebook makes no request.

In [ ]:
treatment = yaml.safe_load(
    (
        EXP / "experiments/phyts_tess/information_treatments/main-fixed-no-prior.yaml"
    ).read_text()
)
print(yaml.safe_dump(treatment, sort_keys=False))
routing = json.loads((WORKFLOW / "agents.json").read_text())
print("Tuner routing:", routing["tune"])
print(
    "Key in THIS notebook process:",
    "set" if os.environ.get("OPENAI_API_KEY") else "not set",
)

## 4. Edit a small demo, not the frozen experiment

One iteration offers one Trial and one Formal round. A failure can prevent a Formal result. One epoch is a ceiling; attempt budgets can stop earlier. The 2/5-minute limits are **not** a total run-time or money cap.

Keep `vram_gib` below the card's physical capacity. Choose `RTX 5090` or `H100`; AMD/Intel are unsupported. Other NVIDIA cards need the explicit support/qualification steps in the tutorial README.

The historic model ID may not be available to your account. An external copy of `agents.json`, selected by `llm_config`, lets you change routing without dirtying source. Preserve the routing schema; do not add API keys. Changing model/provider creates a new treatment.

In [ ]:
settings = DemoSettings(
    version="siderius-tess-tutorial-v1",
    infra_checkout=INFRA,
    data_dir=DATA,
    workspace=HOME / "runs/tess-demo-001",  # must not exist
    run_name="tess_demo_001",
    gpu="RTX 5090",  # or H100
    iterations=1,
    epochs=1,
    trial_minutes=2,
    formal_minutes=5,
    vram_gib=8,
    composition=None,
    llm_config=None,
)
print(settings.model_dump_json(indent=2))
print("Command that the validated settings describe:")
print(shlex.join(build_command(settings)))

### Optional: save editable settings and routing outside source

Set `WRITE_CONFIG = True` once. This creates new files, never overwrites them. To change a provider/model, edit the external routing JSON and set `llm_config` in the external demo JSON to that absolute path. The terminal preview validates it. Reuse a fresh workspace for each changed run.

In [ ]:
WRITE_CONFIG = False
config_path = HOME / "tess-demo.json"
routing_path = HOME / "agents.json"
if WRITE_CONFIG:
    HOME.mkdir(parents=True, exist_ok=True)
    with config_path.open("x") as stream:
        stream.write(settings.model_dump_json(indent=2))
    with routing_path.open("x") as stream:
        json.dump(routing, stream, indent=2)
    print("Created:", config_path, routing_path)
else:
    print("Read-only. Enable WRITE_CONFIG when the paths/settings are ready.")

## 5. Preview and launch from the terminal

First run the preview; inspect its source revisions, composition fingerprint, required key names and argv. Then add `--launch`. The notebook only prints commands.

The preview requires clean exp/infra source and the pinned environments. It resolves trusted task code but calls no provider and runs no training. Launch additionally verifies both data populations, checks the GPU and executes a CUDA allocation. The run stays under your ordinary user; do not use sudo.

In [ ]:
run_command = [
    "bash",
    str(EXP / "tutorials/paper/scripts/run.sh"),
    "--config",
    str(config_path),
]
print("PREVIEW:", shlex.join(run_command))
print("LAUNCH (paid API calls and training):", shlex.join(run_command + ["--launch"]))

## 6. Optional: change the task package itself

A task is more than its composition YAML: the relative references need their sibling declarations, plugins and runtime files. Copy the **whole package**. The example below appends an explicit output-unit clarification to its task description; it leaves tensor shapes, splits and metrics unchanged, but still creates a new composition identity.

After running this exercise, set `composition` in external `tess-demo.json` to the printed path, choose a new run name/workspace and preview again. For a different split, input length or metric, update the runtime and tests as well. In particular, editing an input-shape sentence alone does not change the 1024-sample tensor producer.

In [ ]:
COPY_TASK = False
if COPY_TASK:
    import shutil

    copy_root = HOME / "custom-tess-task"
    shutil.copytree(
        TASK, copy_root, ignore=shutil.ignore_patterns("__pycache__", "*.pyc")
    )
    config_file = copy_root / "declared/task_config.yaml"
    declaration = yaml.safe_load(config_file.read_text())
    declaration["task_description"] += (
        "\nReturn rotation frequency in cycles per day.\n"
    )
    config_file.write_text(yaml.safe_dump(declaration, sort_keys=False))
    print("New composition:", copy_root / "compositions/rotation_regression.yaml")
else:
    print("Original task unchanged. Enable COPY_TASK to create an external variant.")

## 7. Inspect results without assuming success

The wrapper writes a sibling `<workspace>.tutorial.json` receipt. The framework writes node records and per-iteration manifests under the workspace. A Trial score is not a Formal score; an absent result is not zero. Inspect status, scope, primary metric, Health and provenance before comparing candidates.

The next cell lists likely records without dumping model code, full prompts or credentials. Open a selected record deliberately. A demo checks the path through the system, not whether it beats a published baseline.

In [ ]:
receipt_path = settings.workspace.with_name(settings.workspace.name + ".tutorial.json")
if receipt_path.is_file():
    receipt = json.loads(receipt_path.read_text())
    print(
        {
            key: receipt[key]
            for key in (
                "kind",
                "scientific_reproduction",
                "infra_revision",
                "exp_revision",
                "composition_fingerprint",
            )
        }
    )
if settings.workspace.is_dir():
    records = sorted(settings.workspace.rglob("run_output_*.json"))
    manifests = sorted(settings.workspace.rglob("*manifest*.json"))
    print(
        "Tuner records:", [str(p.relative_to(settings.workspace)) for p in records[:20]]
    )
    print(
        "Manifests:", [str(p.relative_to(settings.workspace)) for p in manifests[:20]]
    )
else:
    print("No run workspace yet. This is expected before launch.")

In [ ]:
# Optional: choose one run_output_*.json from the list above.
SELECTED_RECORD = None  # e.g. settings.workspace / ".../run_output_....json"
if SELECTED_RECORD is not None:
    import pandas as pd
    from agent.schemas.hyperparam_tuning import HyperparamTuningOutput

    output = HyperparamTuningOutput.model_validate_json(
        Path(SELECTED_RECORD).read_text()
    )
    display(
        pd.DataFrame(
            [
                {
                    "round": record.round_index,
                    "role": "Trial" if record.is_trial else "Formal",
                    "status": record.status,
                    "metric": record.metric_result.metric_id
                    if record.metric_result
                    else None,
                    "score": record.metric_result.scalar
                    if record.metric_result
                    else None,
                    "health_checks": len(record.health_gate_results),
                }
                for record in output.all_records
            ]
        )
    )
else:
    print(
        "Select a saved tuner record after a run; no scores are invented before then."
    )

## 8. Where the paper run and sandbox differ

For the six-hour RTX 5090 experiment, use the original `experiments/phyts_tess/main_fixed_workflow/launch.sh` with `--arm no-prior`, as shown in the tutorial README. Its immutable deadline and resume behavior are not part of this short demo adapter. It does not read demo settings JSON. The current source pin includes fixes after historical runs; exact archived replay requires the run's own pinned source and checkpoints.

Fixed workflow does not expose an external coding-agent shell. It still executes generated model code in subprocesses. This is **not an OS security sandbox**: readable files and validation targets are not made private from arbitrary code under the same user. Task declarations, process resource controls, filesystem permissions and private-evaluator isolation are different boundaries.

Changing NVIDIA GPU is a new hardware treatment; the same time budget can explore fewer or more models. AMD/Intel GPU support is not provided. Preserve the data split, record changed settings, and compare scores only over the same population.